# P2W1D3: Data Infrastructure for AI

Isi dan tujuannya sama persis dengan `README.md` (vector database, embedding, setup & indexing, insert & query, integrasi ke aplikasi). Baca dulu `README.md` (terutama "Konsep Guided Hands-on" dan "Hands-On-Lab") sebelum mengerjakan notebook ini.

Vector database yang dipakai di notebook ini adalah **Pinecone** -- butuh API key dari [app.pinecone.io](https://app.pinecone.io/). OpenAI API key juga dibutuhkan untuk membuat embedding. Salin `.env.example` menjadi `.env`, lalu isi kedua API key di file `.env`.

**Biaya:** OpenAI API dipanggil setiap kali `embed_text(...)` dijalankan (8 produk + beberapa query). Estimasi total biaya untuk seluruh notebook jauh di bawah USD 1. Pinecone Starter gratis untuk skala latihan ini.


## 1. Install Package

In [1]:
# openai: SDK resmi untuk membuat embedding dengan OpenAI API
# pinecone: SDK resmi untuk setup index, upsert, dan query vector database Pinecone
# python-dotenv: membaca kredensial OpenAI dan Pinecone dari file .env

%pip install -q openai pinecone python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Kredensial lewat file `.env`

Buat file `.env` di folder yang sama dengan notebook ini (lihat `.env.example`), lalu isi
kredensial milikmu sendiri. Kode hanya **membaca** nilainya — key tidak pernah ditulis di notebook.

Isi minimal `.env`:

```
OPENAI_API_KEY=...         # untuk membuat embedding
PINECONE_API_KEY=...       # untuk vector database Pinecone
```

In [2]:
import os
import time
from dotenv import load_dotenv

import openai
from pinecone import Pinecone, ServerlessSpec

load_dotenv(dotenv_path=".env", override=True)

# --- Embedding model ---
openai_api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY belum diisi di file .env.")

openai_client = openai.OpenAI(api_key=openai_api_key)
EMBEDDING_MODEL = "text-embedding-3-small"  # 1536 dimensi


def embed_text(text: str) -> list[float]:
    """Ubah satu string teks menjadi embedding vector (1536 dimensi)."""
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


# --- Vector database: Pinecone ---
pinecone_api_key = os.getenv("PINECONE_API_KEY", "").strip()
if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY belum diisi di file .env.")

pc = Pinecone(api_key=pinecone_api_key)

print(f"Setup selesai. Embedding model: {EMBEDDING_MODEL}")

Setup selesai. Embedding model: text-embedding-3-small


In [14]:
# Delapan produk gear lari elite di katalog RunVerse, identik dengan tabel di README.md (Studi Kasus).
PRODUCTS = {
    "prod-coros-pace3": {
        "text": "COROS PACE 3, GPS running watch ringan (30 gram) dengan baterai tahan hingga 24 hari mode biasa, akurasi pace dan jarak yang konsisten, favorit pelari kompetitif untuk pantau data latihan harian.",
        "category": "jam_gps",
    },
    "prod-garmin-fr965": {
        "text": "Garmin Forerunner 965, GPS watch dengan layar AMOLED terang, fitur training readiness dan running dynamics lengkap, cocok untuk pelari elite yang butuh analisis latihan mendalam.",
        "category": "jam_gps",
    },
    "prod-adidas-adizero-pro3": {
        "text": "Adidas Adizero Adios Pro 3, sepatu lari racing dengan carbon plate dan midsole Lightstrike Pro, didesain untuk mengejar catatan waktu terbaik di lomba marathon.",
        "category": "sepatu",
    },
    "prod-puma-fastr-elite2": {
        "text": "PUMA Fast-R Nitro Elite 2, sepatu racing dengan NITROFOAM Elite dan carbon plate, dirancang aerodinamis untuk pelari elite yang mengejar personal best di jarak jauh.",
        "category": "sepatu",
    },
    "prod-nike-alphafly3": {
        "text": "Nike Alphafly 3, sepatu racing dengan ZoomX foam dan carbon plate ganda, salah satu sepatu paling banyak dipakai pelari elite dunia untuk lomba marathon besar.",
        "category": "sepatu",
    },
    "prod-2xu-compression": {
        "text": "2XU Compression Race Tights, celana kompresi dengan teknologi PWX yang membantu sirkulasi darah dan mengurangi kelelahan otot saat lari jarak jauh.",
        "category": "apparel",
    },
    "prod-nakedrunning-band": {
        "text": "Naked Running Band, hydration belt minimalis tanpa bouncing yang muat beberapa gel dan handphone, favorit pelari elite untuk lomba marathon supaya tidak perlu berhenti di water station.",
        "category": "aksesori",
    },
    "prod-theragun-mini": {
        "text": "Theragun Mini, alat pemulihan otot portable dengan percussive therapy, dipakai pelari elite untuk pemulihan otot cepat setelah sesi latihan berat.",
        "category": "recovery",
    },
}

## 3. Setup & Indexing Vector Database

Buat index Pinecone sesuai **Kontrak Index** pada **Konsep 2** di README: `INDEX_NAME`, `DIMENSION = 1536`, `METRIC = "cosine"`. Gunakan nilai yang sama persis di bagian Insert dan Query nanti -- kalau dimension yang dibuat di sini tidak sama dengan dimensi output `embed_text()`, insert atau query di bagian berikutnya akan gagal.

> Bagian ini hanya membuat atau membuka index. Data `PRODUCTS` belum dimasukkan, sehingga `total_vector_count=0` adalah hasil yang benar untuk index baru. Jumlahnya menjadi 8 setelah Bagian 4 dijalankan.


In [12]:
INDEX_NAME = "rahoel-02"
DIMENSION = 1536
METRIC = "cosine"
NAMESPACE = "rahoel-02"

if not pc.indexes.exists(INDEX_NAME):
    pc.indexes.create(
        name=INDEX_NAME,
        vector_type="dense",
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# Index serverless butuh beberapa detik untuk siap dipakai setelah dibuat.
ready_deadline = time.time() + 120
while not pc.indexes.describe(INDEX_NAME).status["ready"]:
    if time.time() >= ready_deadline:
        raise TimeoutError(f"Index {INDEX_NAME!r} belum siap setelah 120 detik.")
    time.sleep(1)

index = pc.index(INDEX_NAME)
stats = index.describe_index_stats()
if stats.dimension != DIMENSION or stats.metric != METRIC:
    raise ValueError(
        f"Kontrak index tidak sesuai. Ditemukan dimension={stats.dimension}, metric={stats.metric}; "
        f"seharusnya dimension={DIMENSION}, metric={METRIC}. Gunakan index baru dengan konfigurasi yang benar."
    )
print(stats)

DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)


## 4. Persiapan & Insert Data Vektor

Lengkapi `upsert_products()`: untuk tiap produk di `PRODUCTS` (**Konsep 3**), buat embedding-nya lewat `embed_text()`, lalu simpan beserta metadata `source`, `category`, dan `text` ke Pinecone. Pakai `id` dari `PRODUCTS` sebagai identifier (bukan ID acak), supaya proses ini aman dijalankan ulang lewat `index.upsert()` tanpa membuat data ganda (idempoten).


In [15]:
def upsert_products(products: dict) -> None:
    """Ubah setiap produk di `products` jadi embedding, lalu simpan ke Pinecone."""
    vectors = []
    for product_id, product in products.items():
        embedding = embed_text(product["text"])
        vectors.append({
            "id": product_id,
            "values": embedding,
            "metadata": {"source": product_id, "category": product["category"], "text": product["text"]},
        })

    index.upsert(vectors=vectors, namespace=NAMESPACE)

def wait_for_vector_count(expected_count: int, timeout_seconds: int = 30):
    """Tunggu sampai statistik Pinecone mencatat jumlah vector yang diharapkan."""
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        stats = index.describe_index_stats()
        namespace_stats = stats.namespaces.get(NAMESPACE)
        actual_count = namespace_stats.vector_count if namespace_stats else 0
        if actual_count == expected_count:
            return stats
        time.sleep(1)
    raise TimeoutError(
        f"Namespace {NAMESPACE!r} belum mencapai {expected_count} vector. "
        f"Jumlah terakhir: {actual_count}."
    )


upsert_products(PRODUCTS)
stats = wait_for_vector_count(len(PRODUCTS))
print(f"Verifikasi berhasil: {stats.namespaces[NAMESPACE].vector_count} produk tersimpan.")

Verifikasi berhasil: 8 produk tersimpan.


## 5. Query Data Vektor

Lengkapi `query_similar(query_text, top_k, category=None)` (**Konsep 4**): ubah `query_text` jadi embedding dengan model **yang sama** seperti saat insert, cari `top_k` produk paling mirip, lalu kembalikan teks, skor, dan metadata-nya. Pertanyaan uji memakai susunan kata dan gaya bahasa yang berbeda dari deskripsi produk. Hands-on dianggap berhasil jika produk target muncul dalam hasil Top-3, karena hasil Top-1 belum tentu selalu yang paling tepat.


In [16]:
def query_similar(query_text: str, top_k: int = 3, category: str = None) -> list[dict]:
    """Cari top_k produk paling mirip dengan query_text. Filter opsional lewat `category`."""
    query_embedding = embed_text(query_text)

    query_kwargs = dict(namespace=NAMESPACE, vector=query_embedding, top_k=top_k, include_metadata=True)
    if category:
        query_kwargs["filter"] = {"category": category}

    results = index.query(**query_kwargs)

    return [
        {"id": m["id"], "score": m["score"], "text": m["metadata"]["text"], "category": m["metadata"]["category"]}
        for m in results["matches"]
    ]

PARAPHRASE_QUESTIONS = [
    "Jam tangan buat pantau data lari harian yang baterainya awet",
    "Sepatu racing yang enteng buat ngejar waktu terbaik di marathon",
    "Celana yang bisa bantu sirkulasi darah pas lari jauh",
]

for question in PARAPHRASE_QUESTIONS:
    print(f"Q: {question}")
    for r in query_similar(question, top_k=3):
        print(f"  [{r['score']:.3f}] {r['id']} ({r['category']}): {r['text']}")
    print()

Q: Jam tangan buat pantau data lari harian yang baterainya awet
  [0.571] prod-coros-pace3 (jam_gps): COROS PACE 3, GPS running watch ringan (30 gram) dengan baterai tahan hingga 24 hari mode biasa, akurasi pace dan jarak yang konsisten, favorit pelari kompetitif untuk pantau data latihan harian.
  [0.500] prod-nakedrunning-band (aksesori): Naked Running Band, hydration belt minimalis tanpa bouncing yang muat beberapa gel dan handphone, favorit pelari elite untuk lomba marathon supaya tidak perlu berhenti di water station.
  [0.460] prod-garmin-fr965 (jam_gps): Garmin Forerunner 965, GPS watch dengan layar AMOLED terang, fitur training readiness dan running dynamics lengkap, cocok untuk pelari elite yang butuh analisis latihan mendalam.

Q: Sepatu racing yang enteng buat ngejar waktu terbaik di marathon
  [0.683] prod-adidas-adizero-pro3 (sepatu): Adidas Adizero Adios Pro 3, sepatu lari racing dengan carbon plate dan midsole Lightstrike Pro, didesain untuk mengejar catatan waktu terbai

Tambahkan satu uji dengan **metadata filter** berdasarkan `category`, untuk kasus ketika pelanggan sudah tahu jenis produk yang dicari.

In [ ]:
# TODO: uji query_similar() dengan metadata filter, misalnya kategori "sepatu"
for r in query_similar("cari sepatu racing yang bagus", top_k=5, category="sepatu"):
    print(f"[{r['score']:.3f}] {r['id']} ({r['category']}): {r['text']}")

## 6. Integrasi ke Aplikasi Sederhana

Lengkapi `search(query_text, top_k, min_score)` (**Konsep 5**): bungkus `query_similar()` jadi satu fungsi pencarian aplikasi, ditambah pengecekan ambang skor kemiripan -- kalau skor hasil teratas di bawah `min_score`, kembalikan pesan "tidak ditemukan" alih-alih memaksakan hasil yang tidak relevan. Uji dengan seluruh `USER_QUESTIONS` di bawah, termasuk satu pertanyaan yang sengaja di luar cakupan 8 produk.

> **Catatan tuning:** nilai `min_score` tidak bersifat universal. Gunakan `0.4` sebagai baseline latihan, lalu bandingkan skor pertanyaan relevan dan pertanyaan di luar cakupan. Naikkan atau turunkan nilainya jika model atau data Anda menghasilkan rentang skor yang berbeda.


In [ ]:
def search(query_text: str, top_k: int = 3, min_score: float = 0.4) -> str:
    """Fungsi pencarian aplikasi: bungkus query_similar(), tangani hasil kurang relevan."""
    results = query_similar(query_text, top_k=top_k)

    # TODO: kalau `results` kosong ATAU skor tertinggi (results[0]["score"]) di bawah `min_score`,
    # return "Maaf, tidak ditemukan produk yang relevan untuk pertanyaan ini."
    ...

    # TODO: kalau lolos pengecekan di atas, format hasil jadi string yang enak dibaca:
    # peringkat, score, id, category, dan potongan teksnya. Gunakan "\n".join(...) pada akhirnya.
    lines = []
    return "\n".join(lines)


USER_QUESTIONS = [
    "Saya butuh jam GPS yang detail buat analisis latihan, bukan cuma pace doang",
    "Ada sepatu racing carbon plate yang cocok buat kejar PB marathon?",
    "Tas pinggang buat bawa gel pas lomba tanpa ribet ke water station",
    "Alat pijat portable buat pemulihan otot abis latihan berat",
    "Bagaimana cara reset password akun saya di RunVerse?",  # di luar cakupan 8 produk
]

for question in USER_QUESTIONS:
    print(f"Q: {question}")
    print(search(question))
    print()

### Refleksi Implementasi

Isi berdasarkan hasil run `search()` pada seluruh `USER_QUESTIONS` di atas.

| Pertanyaan | Produk Teratas yang Ditemukan | Relevan? (Ya/Belum/Tidak Ditemukan) | Catatan |
| --- | --- | --- | --- |
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |

**Catatan tuning `min_score`:** Nilai berapa yang akhirnya Anda pakai, dan bagaimana Anda menentukannya dari hasil run di atas?
